# 03 — Candidate-path reduction and reduced QUBO

Complete admissible paths are generated classically and the binary problem selects one path per service under the shared OEC resource constraints.

In [ ]:
from __future__ import annotations

import pandas as pd

from oec_qaoa import canonical_c2
from oec_qaoa.edge_qubo import build_edge_qubo
from oec_qaoa.ilp import solve_reference_ilp
from oec_qaoa.path_qubo import build_path_qubo, solve_path_qubo_exact
from oec_qaoa.path_reduction import (
    candidate_sets,
    decomposition_gap,
    solve_path_selection,
)

instance = canonical_c2()
reference = solve_reference_ilp(instance)

print(f"Reference objective: {reference.objective:.3f}")

## Routing instance C2

C2 gives both services two relay choices. `h0` can be processed only on `R1`, while `h1` can be processed on either relay. The zero-ground-processing optimum routes `h0` through `R1` and `h1` through `R2`.

In [ ]:
rows = []
all_paths = candidate_sets(instance)

for service_name, paths in all_paths.items():
    for path in paths:
        rows.append(
            {
                "service": service_name,
                "path": path.name,
                "objective": path.objective,
                "edges": " -> ".join(path.edges),
            }
        )

paths_table = pd.DataFrame(rows)
paths_table

Candidate paths are ranked by ground-processing objective, path length and a deterministic edge-name tie break.

## Candidate-set size

Let `P_h^(K)` be the first `K` paths retained for service `h`. In C2, `K=1` is jointly infeasible because both locally best paths use `R1`; `K=2` recovers the global optimum.

In [ ]:
records = []

for k in (1, 2):
    paths = candidate_sets(instance, k=k)
    solution = solve_path_selection(instance, paths)

    records.append(
        {
            "K": k,
            "number_of_path_variables": sum(len(v) for v in paths.values()),
            "feasible": solution.success,
            "objective": solution.objective if solution.success else None,
            "decomposition_gap": (
                decomposition_gap(reference.objective, solution)
                if solution.success
                else None
            ),
            "selected_paths": (
                solution.selected_paths if solution.success else None
            ),
        }
    )

pd.DataFrame(records)

For feasible restricted models the decomposition gap is

\[
\Delta_K=z_{\mathrm{path}}^*(K)-z_{\mathrm{edge}}^*.
\]

Feasibility and objective gap are reported separately.

## Reduced path-QUBO

The semantic variable `x_hk` selects candidate path `k` for service `h`, with

\[
\sum_k x_{hk}=1.
\]

Source, destination and flow consistency are satisfied by construction; shared transmission, storage, processing and energy constraints remain in the optimization.

In [ ]:
paths_k2 = candidate_sets(instance, k=2)
path_exact = solve_path_selection(instance, paths_k2)
path_qubo = build_path_qubo(instance, paths_k2)
path_qubo_solution = solve_path_qubo_exact(path_qubo)

resource_comparison = pd.DataFrame(
    [
        {
            "formulation": "direct edge QUBO",
            "semantic_variables": build_edge_qubo(instance).semantic_count,
            "logical_variables": len(build_edge_qubo(instance).variable_names),
            "quadratic_couplings": len(build_edge_qubo(instance).quadratic),
        },
        {
            "formulation": "path QUBO, K=2",
            "semantic_variables": path_qubo.semantic_count,
            "logical_variables": len(path_qubo.variable_names),
            "quadratic_couplings": len(path_qubo.quadratic),
        },
    ]
)
resource_comparison

In [ ]:
print(f"Reduced exact objective: {path_exact.objective:.3f}")
print(f"Path-QUBO energy: {path_qubo_solution.energy:.3f}")
print(f"Exact selected paths: {path_exact.selected_paths}")
print(f"QUBO selected paths: {path_qubo_solution.selected_paths}")

## Equivalence check

For `K=2`, the reduced constrained model and the path-QUBO recover the direct OEC optimum.

In [ ]:
assert path_exact.success
assert path_qubo_solution.success
assert path_exact.objective == reference.objective == 0.0
assert abs(path_qubo_solution.energy - path_exact.objective) < 1e-9
assert path_exact.selected_paths == {
    "h0": "P_h0_0",
    "h1": "P_h1_1",
}
assert path_qubo_solution.selected_paths == {
    "h0": ("P_h0_0",),
    "h1": ("P_h1_1",),
}

print("C2 path-reduction and path-QUBO validation: PASS")

## Result

C2 shows that path preprocessing can reduce the semantic decision space while preserving the optimum, provided that the retained candidate set contains a jointly feasible combination.